In [1]:
from astropy.io import fits
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset

In [2]:
# Definir el Dataset
class SpectraDataset(Dataset):
    def __init__(self, flux_path, wavelength_path, redshift_path, total, num_points):
        """
        Parámetros:
          flux_path, wavelength_path, redshift_path: rutas a los archivos .dat (ya normalizados para flujo y longitud de onda)
          total: número total de espectros
          num_points: número de puntos por espectro
        """
        self.flux = np.memmap(flux_path, dtype="float32", mode="r", shape=(total, num_points))
        self.wavelength = np.memmap(wavelength_path, dtype="float32", mode="r", shape=(total, num_points))
        self.redshift = np.memmap(redshift_path, dtype="float32", mode="r", shape=(total,))
        self.total = total
        self.num_points = num_points

    def __len__(self):
        return self.total

    def __getitem__(self, idx):
        # Leer la muestra individualmente
        flux_sample = self.flux[idx, :].copy()
        wave_sample = self.wavelength[idx, :].copy()
        # Combinar en un array de forma (2, num_points)
        X = np.stack([flux_sample, wave_sample], axis=0)
        # Leer el redshift
        y = self.redshift[idx]
        # Convertir a tensores de PyTorch
        X_tensor = torch.tensor(X, dtype=torch.float32)
        # Se aplica unsqueeze para que el target tenga forma (1,)
        y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        return X_tensor, y_tensor

# Parámetros y rutas
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

data_dir = "data"
num_points = 5000

# Parámetros de tamaño
big_total = 595472     # Número de espectros en bigtraining
small_total = 4528     # Número de espectros en smalltraining

# Rutas de los archivos bigtraining (datos normalizados)
big_flux_norm_path = os.path.join(data_dir, "spectra_data_bigtraining_flux_norm.dat")
big_wavelength_norm_path = os.path.join(data_dir, "spectra_data_bigtraining_wavelength_norm.dat")
big_redshift_path = os.path.join(data_dir, "spectra_data_bigtraining_redshift_norm.dat")

# Rutas de los archivos smalltraining (datos normalizados)
small_flux_norm_path = os.path.join(data_dir, "spectra_data_smalltraining_flux_norm.dat")
small_wavelength_norm_path = os.path.join(data_dir, "spectra_data_smalltraining_wavelength_norm.dat")
small_redshift_path = os.path.join(data_dir, "spectra_data_smalltraining_redshift_norm.dat")

# Dividir el conjunto bigtraining en train y test (84% train, 16% test)
indices_big = np.arange(big_total)
big_train_indices, big_test_indices = train_test_split(indices_big, test_size=0.16, random_state=42)

# Crear los Datasets con los datos ya normalizados
big_dataset_norm = SpectraDataset(big_flux_norm_path, big_wavelength_norm_path, big_redshift_path, big_total, num_points)
small_dataset_norm = SpectraDataset(small_flux_norm_path, small_wavelength_norm_path, small_redshift_path, small_total, num_points)

# Aplicar los mismos índices para los subconjuntos de bigtraining
big_train_dataset_norm = Subset(big_dataset_norm, big_train_indices)
big_test_dataset_norm = Subset(big_dataset_norm, big_test_indices)

# Conjunto de entrenamiento final normalizado: 84% de bigtraining + todo smalltraining
train_dataset = ConcatDataset([big_train_dataset_norm, small_dataset_norm])
test_dataset = big_test_dataset_norm  # El test proviene solo de bigtraining

# Crear DataLoaders (puedes ajustar batch_size y num_workers según convenga)
train_loader = DataLoader(train_dataset, shuffle=True)
test_loader = DataLoader(test_dataset, shuffle=False)

print("Listo para entrenar a partir de los datos normalizados y usando memmap de forma eficiente.")

Usando dispositivo: cuda
Listo para entrenar a partir de los datos normalizados y usando memmap de forma eficiente.


In [3]:
# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Número de puntos en cada espectro
num_points = 5000

# Definir el modelo CNN en PyTorch con dropout para regularización
class CNN(nn.Module):
    def __init__(self, num_points):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        # Tras 3 max pooling, la dimensión se reduce en un factor de 8
        conv_output_size = num_points // 8
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * conv_output_size, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),  # Regularización con Dropout
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

modelCNN = CNN(num_points).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(modelCNN.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

Usando dispositivo: cuda


In [ ]:
# Entrenar el modelo
num_epochs = 5
for epoch in range(num_epochs):
    modelCNN.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = modelCNN(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_dataset)
    
    # Evaluación en el conjunto de validación
    modelCNN.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = modelCNN(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(test_dataset)
    
    # Actualizar la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.10f}", flush=True)

# Evaluación final en el conjunto de prueba utilizando MAE
mae_loss = nn.L1Loss()
modelCNN.eval()
test_mae = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = modelCNN(batch_X)
        loss = mae_loss(outputs, batch_y)
        test_mae += loss.item() * batch_X.size(0)
test_mae /= len(test_dataset)
print(f"Error absoluto medio en el conjunto de prueba: {test_mae:.4f}")

# Guardar los parámetros del modelo
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': modelCNN.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}
torch.save(checkpoint, 'storage/modelCNN_UPD_100ktest.pth')

In [ ]:
# Seguir entrenando el modelo
checkpoint = torch.load('storage/modelCNN_UPD_100ktest.pth', weights_only=False)
modelCNN.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch']

# Definir el número total de epochs que deseas entrenar
num_epochs = 10

# Continuar el entrenamiento desde el epoch donde se quedó
for epoch in range(start_epoch, num_epochs):
    modelCNN.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = modelCNN(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_dataset)

    # Evaluación en el conjunto de validación
    modelCNN.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = modelCNN(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(test_dataset)

    # Actualizar la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.10f}")

# Evaluación final en el conjunto de prueba utilizando MAE
mae_loss = nn.L1Loss()
modelCNN.eval()
test_mae = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = modelCNN(batch_X)
        loss = mae_loss(outputs, batch_y)
        test_mae += loss.item() * batch_X.size(0)
test_mae /= len(test_dataset)
print(f"Error absoluto medio en el conjunto de prueba: {test_mae:.4f}")

# Guardar los parámetros del modelo y el scaler
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': modelCNN.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}
torch.save(checkpoint, 'storage/modelCNN_UPD_100ktest.pth')

In [ ]:
# Obtener la lista inicial de archivos FITS
folder_path = r'spectrums'
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]
files = random.sample(files, len(files))
file_path = os.path.join(folder_path, files[0])

checkpoint = torch.load('storage/modelCNN_UPD_100ktest.pth', weights_only=False)
modelCNN.load_state_dict(checkpoint['model_state_dict'])
with open('extra/scaler_modelCNN_UPD_100ktest.pkl', 'rb') as f:
    scaler = pickle.load(f)


with fits.open(file_path) as hdul:
    test_flux = hdul[1].data["flux"]
    test_loglam = hdul[1].data["loglam"]
    test_redshift = hdul[2].data["Z"][0]  # Asumiendo que Z es un array y queremos el primer valor

test_wavelength = 10 ** test_loglam

def expand_points(wavelength, flux, target_count=5000):
    # Convertir a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Calcular las diferencias absolutas entre puntos consecutivos
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Obtener los índices ordenados de mayor a menor diferencia
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insertar nuevos puntos utilizando los índices ordenados
    while len(wl) < target_count:
        # Se recorre la lista de índices en orden descendente para evitar problemas con el reordenamiento
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Calcular la interpolación lineal entre el punto idx y el siguiente
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insertar el nuevo punto en la posición correspondiente
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

test_wavelength, test_flux = expand_points(test_wavelength, test_flux, target_count=num_points)

# Preprocesamiento para el modelo
# Canal 0: flux; Canal 1: wavelength
input_data = np.stack([test_flux, test_wavelength], axis=0)  # (2, 5000) 2 canales de tamaño 5000
input_data = input_data.reshape(1, 2, num_points)            # (1, 2, 5000) 1 muestra, 2 canales, 5000 puntos/canal

# Normalización
nsamples, nchannels, npoints = input_data.shape # Guardar dimensionalidad inicial
input_flat = input_data.reshape(nsamples, -1) # Aplanar/concatenar (1, 10000)
input_scaled = scaler.transform(input_flat) # Normalizar
input_scaled = input_scaled.reshape(nsamples, nchannels, npoints) # Recuperar dimensionalidad incial

# Convertir a tensor
input_tensor = torch.tensor(input_scaled, dtype=torch.float32)
input_tensor = input_tensor.to(device)

# Evaluar el modelo
with torch.no_grad():
    predicted_redshift = modelCNN(input_tensor)

print("Redshift real:", test_redshift)
print("Redshift predicho:", predicted_redshift.item())

plt.figure(figsize=(12, 6))
plt.plot(test_wavelength, test_flux, label="Test Espectro")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Test)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()